# Recursion and its alternatives

Instructor exploration notebook for Week 09. The examples compare recursive solutions with direct formulas, loops, memoization, and Python's built-in sort. The central question is not whether recursion is good or bad: it is whether the problem's structure makes recursive decomposition clear, and what costs recursion introduces.

Run the cells from top to bottom. Small examples make it easier to check correctness; counters make repeated work visible without relying on machine-dependent timing.

## 1. Sum the integers from 0 through n

For a nonnegative integer `n`, compare the closed form, a loop, and a recursive definition. In the recursive version, the smaller problem is the sum through `n - 1`, and `n` is the work added at this step. The base case is `n == 0`.

In [ ]:
def sum_formula(n):
    if n < 0:
        raise ValueError("n must be nonnegative")
    return n * (n + 1) // 2

def sum_loop(n):
    if n < 0:
        raise ValueError("n must be nonnegative")
    total = 0
    for value in range(n + 1):
        total += value
    return total

def sum_recursive(n):
    if n < 0:
        raise ValueError("n must be nonnegative")
    if n == 0:                 # base case
        return 0
    return n + sum_recursive(n - 1)  # smaller subproblem

sum_comparison = []
for n in range(8):
    row = {
        "n": n,
        "closed form": sum_formula(n),
        "loop": sum_loop(n),
        "recursive": sum_recursive(n),
    }
    sum_comparison.append(row)

for row in sum_comparison:
    print(row)

All three return the same result. The formula uses a constant number of arithmetic operations (with the usual caveat that very large integers take more time to manipulate). The loop and recursive version each do work proportional to `n`. The recursive version also creates about `n` pending function calls, so a mathematically valid definition can exceed the language's recursion limit for large `n`. Ask students to identify the base case, progress toward it, and what would happen if `n` did not decrease.

## 2. Fibonacci: repeated subproblems

Use `F(0) = 0`, `F(1) = 1`, and `F(n) = F(n - 1) + F(n - 2)`. Plain recursion mirrors the definition, but it recomputes values. A loop and memoization preserve the same recurrence while avoiding most repeated calculations.

In [ ]:
def fibonacci_recursive(n):
    if n < 0:
        raise ValueError("n must be nonnegative")
    if n < 2:
        return n
    return fibonacci_recursive(n - 1) + fibonacci_recursive(n - 2)

def fibonacci_loop(n):
    if n < 0:
        raise ValueError("n must be nonnegative")
    previous, current = 0, 1
    for _ in range(n):
        previous, current = current, previous + current
    return previous

def fibonacci_memoized(n, cache=None):
    if n < 0:
        raise ValueError("n must be nonnegative")
    if cache is None:
        cache = {0: 0, 1: 1}
    if n not in cache:
        cache[n] = fibonacci_memoized(n - 1, cache) + fibonacci_memoized(n - 2, cache)
    return cache[n]

[(n, fibonacci_recursive(n), fibonacci_loop(n), fibonacci_memoized(n)) for n in range(11)]

In [ ]:
def fibonacci_recursive_counted(n, calls):
    calls[0] += 1
    if n < 2:
        return n
    return fibonacci_recursive_counted(n - 1, calls) + fibonacci_recursive_counted(n - 2, calls)

def fibonacci_memoized_counted(n, cache, calls):
    calls[0] += 1
    if n not in cache:
        cache[n] = (fibonacci_memoized_counted(n - 1, cache, calls) +
                    fibonacci_memoized_counted(n - 2, cache, calls))
    return cache[n]

comparison = []
for n in (5, 10, 20, 30):
    plain_calls = [0]
    plain_value = fibonacci_recursive_counted(n, plain_calls)
    memo_calls = [0]
    memo_value = fibonacci_memoized_counted(n, {0: 0, 1: 1}, memo_calls)
    assert plain_value == memo_value
    comparison.append((n, plain_value, plain_calls[0], memo_calls[0]))
comparison

The recursive call count grows rapidly because the same subproblems appear in many branches. Memoization stores answers, reducing the amount of work to linear in `n`, though it still uses recursive calls and a cache. The iterative version also takes linear time and constant extra space. Memoization is especially useful when recursive subproblems overlap; it does not automatically make every recursive algorithm efficient. Both recursive versions can hit the call-depth limit for sufficiently large `n`, even when the number of distinct memoized values is small.

## 3. Sorting: bubble sort, merge sort, and `sorted`

Bubble sort repeatedly swaps adjacent out-of-order values. Merge sort splits the list, recursively sorts each half, then merges them. Python's built-in `sorted` is the practical default: it is highly optimized and stable. Compare correctness and comparison counts first; optional timings vary with hardware, Python version, input shape, and measurement noise.

In [ ]:
def bubble_sort(values):
    result = list(values)
    comparisons = 0
    for end in range(len(result) - 1, 0, -1):
        swapped = False
        for index in range(end):
            comparisons += 1
            if result[index] > result[index + 1]:
                result[index], result[index + 1] = result[index + 1], result[index]
                swapped = True
        if not swapped:          # already sorted: stop early
            break
    return result, comparisons

def merge_sort(values):
    values = list(values)
    if len(values) <= 1:       # base case
        return values, 0
    middle = len(values) // 2
    left, left_comparisons = merge_sort(values[:middle])
    right, right_comparisons = merge_sort(values[middle:])

    merged = []
    i = j = comparisons = 0
    while i < len(left) and j < len(right):
        comparisons += 1
        if left[i] <= right[j]:   # <= keeps merge sort stable
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged, left_comparisons + right_comparisons + comparisons

sample = [8, 3, 7, 4, 9, 2, 6, 5]
bubble_result, bubble_comparisons = bubble_sort(sample)
merge_result, merge_comparisons = merge_sort(sample)
builtin_result = sorted(sample)
assert bubble_result == merge_result == builtin_result
{"input": sample, "bubble": (bubble_result, bubble_comparisons),
 "merge": (merge_result, merge_comparisons), "built_in": builtin_result}

In [ ]:
from random import Random

rng = Random(2043)
sizes = (8, 16, 32, 64, 128)
sort_comparison_counts = []
for size in sizes:
    data = [rng.randrange(0, 1000) for _ in range(size)]
    _, bubble_count = bubble_sort(data)
    _, merge_count = merge_sort(data)
    assert bubble_sort(data)[0] == merge_sort(data)[0] == sorted(data)
    sort_comparison_counts.append((size, bubble_count, merge_count))
sort_comparison_counts

Bubble sort has quadratic worst-case time; merge sort has `O(n log n)` time and uses additional memory. Both are recursive/iterative examples of broader design choices: recursion describes merge sort's divide-and-conquer structure naturally, while bubble sort's repeated adjacent passes fit a loop. Built-in sorting is usually the right choice in actual Python programs. The educational implementations let us see the algorithmic structures that the library hides.

### Optional timing cell

Keep the size small enough that bubble sort finishes quickly. Use repeated measurements and treat timings as observations about this environment, not proofs of complexity.

In [ ]:
from timeit import timeit

timing_data = [rng.randrange(0, 10_000) for _ in range(500)]
timings_seconds = {
    "bubble sort": timeit(lambda: bubble_sort(timing_data), number=1),
    "merge sort": timeit(lambda: merge_sort(timing_data), number=20) / 20,
    "built-in sorted": timeit(lambda: sorted(timing_data), number=100) / 100,
}
timings_seconds

## Wrap-up prompts

1. For each recursive function, point to the base case and explain why each call gets closer to it.
2. Which example exposes repeated subproblems? Which one divides the input into independent pieces?
3. Where does recursion make the structure easier to express, and where does it add avoidable overhead or a depth limit?
4. How would you adapt Week 09's small-case enumeration habit to check these implementations? Which small inputs are most revealing?
5. What evidence does a timing experiment provide, and what does it not establish?